# 10 · Error analysis — where verification helps, and where it hurts

**Purpose.** Break the selected pipeline's decisions down by category, object size, detector confidence, image brightness (low-light proxy) and co-occurrence-driven false positives; show galleries of correct rejections, omissions and over-rejections.

| | |
|---|---|
| **Proposal objective** | Expected contribution 4 |
| **Inputs** | detection + verifier caches, `results/data/cooccurrence_top10.json` |
| **Outputs** | `results/error_analysis/*.csv`, figures |
| **Runtime (T4)** | ≈5 min |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess  # paths, sys.path and shell commands (git clone, pip)
IN_COLAB = "google.colab" in sys.modules

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"
BRANCH = "main"  # git branch to take the latest code from (change it if you pushed to another branch)

if IN_COLAB:
    from google.colab import drive  # Colab only: mount Google Drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, DRIVE_PROJECT], check=True)
    elif os.path.isdir(os.path.join(DRIVE_PROJECT, ".git")):   # pick up the latest pushed code (never touches your notebooks/results)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "checkout", "FETCH_HEAD", "--", "src", "configs", "requirements.txt", "tools", "app"], check=False)
    PROJECT_ROOT = DRIVE_PROJECT
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab already has torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt  # json files, numpy arrays, pandas tables, matplotlib plots
from dvd.config import load_config, save_selected, results_dir  # load configs/default.yaml, save tuned values, results folders
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache  # seeding, JSON helpers and the resumable JSONL cache
from dvd import paths  # cache-file path helpers

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch  # PyTorch: GPU check and memory statistics
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_annotations, fetch_images, CocoData, load_splits  # COCO annotations, image download, splits
prepare_annotations(cfg)   # annotation json only (the 241 MB zip is cached on Drive)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
fetch_images(cfg, HELDOUT + TEST)   # only these images (a few hundred files), not the whole 778 MB val2017 zip
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.experiments import SplitData, available_methods, masks_for, PROPOSED  # methods over identical candidates: sweeps, operating points, evaluation, costs
from dvd.analysis import candidate_table, mean_brightness  # per-candidate error-analysis tables and co-occurrence statistics
from dvd.viz import crop_gallery  # plotting helpers (detections overlay, PR frontier, crop gallery)
from tqdm.auto import tqdm  # progress bars
T = SplitData(cfg, coco, "test", TEST)
M = available_methods(cfg, T)[PROPOSED]
tl, th = cfg["selected"]["tau_low"], cfg["selected"]["tau_high"]
accept = masks_for(M, T, {"tau_low": tl, "tau_high": th})
cooc = {int(k): set(v) for k, v in load_json(results_dir(cfg, "data") / "cooccurrence_top10.json").items()}
bright = {i: mean_brightness(coco.image(i)) for i in tqdm(TEST, desc="brightness")}
df = pd.DataFrame(candidate_table(T.det, T.labels, accept, T.gts, M["verdicts"], M["p_yes"], cooc, bright))
df = df[(df.score >= tl) & (df.outcome != "ignore")]          # reference pool: what the detector outputs at tau_low
df["band"] = np.where(df.score >= th, "auto-accept (c >= tau_high)", "verified band")
print(df.outcome.value_counts())

In [ ]:
def rates(g):
    # Per group: candidate count, FP/TP counts, share of FPs removed and share of TPs lost.
    fp, tp = g[g.is_fp], g[g.is_tp]
    return pd.Series({"candidates": len(g), "FP": len(fp), "TP": len(tp),
                      "FP removed (correct rejection)": (~fp.accepted).mean() if len(fp) else np.nan,
                      "TP lost (over-rejection)": (~tp.accepted).mean() if len(tp) else np.nan})
out = results_dir(cfg, "error_analysis")
by_cat = df.groupby("label").apply(rates).sort_values("FP", ascending=False); by_cat.to_csv(out / "by_category.csv")
by_cat.head(20)

In [ ]:
df["conf_bin"] = pd.cut(df.score, [0, .3, .4, .5, .6, .8, 1.0])
df["brightness_bin"] = pd.qcut(df.brightness, 3, labels=["dark", "medium", "bright"])
for col in ["size", "conf_bin", "brightness_bin", "cooccurrence_fp", "band"]:
    t = df.groupby(col, observed=True).apply(rates); t.to_csv(out / f"by_{col}.csv")
    print(f"\n=== by {col} ==="); display(t)

### Where does the verifier help most / hurt most?

In [ ]:
k = by_cat[by_cat.FP >= 20]
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
k.sort_values("FP removed (correct rejection)").tail(15)["FP removed (correct rejection)"].plot.barh(ax=ax[0], title="Highest FP-removal categories (>=20 FPs)")
by_cat[by_cat.TP >= 20].sort_values("TP lost (over-rejection)").tail(15)["TP lost (over-rejection)"].plot.barh(ax=ax[1], color="C3", title="Highest over-rejection categories (>=20 TPs)")
plt.tight_layout(); plt.savefig(results_dir(cfg, "figures") / "error_by_category.png", dpi=150, bbox_inches="tight"); plt.show()

### Galleries (crops exactly as the verifier saw them)

In [ ]:
def gallery_items(outcome, n=12):
    # Random sample of candidates with a given outcome, formatted for crop_gallery.
    pool = df[(df.outcome == outcome) & (df.band == "verified band")]
    s = pool.sample(min(n, len(pool)), random_state=0)
    return [{"image_id": r.image_id, "box": r.box, "label": r.label, "score": r.score, "p_yes": r.p_yes} for r in s.itertuples()]
for oc in ["correct_rejection", "omission", "over_rejection"]:
    crop_gallery(gallery_items(oc), coco.image, cfg["verifier"]["crop_margin"], title=oc.replace("_", " "))
    plt.savefig(results_dir(cfg, "figures") / f"gallery_{oc}.png", dpi=120, bbox_inches="tight"); plt.show()